# Temporal Analysis: Sahel Security Analysis
---
This notebook analyzes the temporal dynamics of conflict in the Sahel region 
(Burkina Faso, Mali, Niger) using ACLED data from 2020 to 2025.

**What we cover in this notebook:**
1. Monthly evolution of incidents by country
2. Monthly fatalities over time
3. Breakdown by event type
4. Anomaly detection (abnormal conflict peaks)
5. Cross-country correlation (do conflicts spread?)
6. Trend model + 6-month forecast

**Data source:** ACLED (Armed Conflict Location & Event Data Project)  
**Period:** January 2020 to March 2025(the maximum allowed for my api)  
**Countries:** Burkina Faso, Mali, Niger

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Load processed data
df = pd.read_csv("../data/processed/acled_processed.csv", parse_dates=["event_date"])

# Create a proper period column for monthly aggregation
df["year_month"] = df["event_date"].dt.to_period("M").astype(str)

print(f"Records loaded  : {len(df):,}")
print(f"Date range      : {df['event_date'].min().date()} -> {df['event_date'].max().date()}")
print(f"Countries       : {list(df['country'].unique())}")

Records loaded  : 23,156
Date range      : 2020-01-01 -> 2025-03-28
Countries       : ['Niger', 'Burkina Faso', 'Mali']


## 1. Monthly Evolution of Incidents
---
We start by aggregating incidents at the **monthly level per country** to observe 
the overall conflict trajectory across the three Sahel nations.

**Key questions:**
- Which country shows the highest conflict intensity?
- Are there visible seasonal patterns?
- When did the major escalations occur?

In [4]:
# --- Monthly aggregation per country
monthly = (
    df.groupby(["year_month", "country"])
    .agg(
        incidents  = ("event_id_cnty", "count"),
        fatalities = ("fatalities",    "sum"),
    )
    .reset_index()
)
monthly["year_month_dt"] = pd.to_datetime(monthly["year_month"])
monthly = monthly.sort_values("year_month_dt")

# --- Plot
fig = px.line(
    monthly,
    x="year_month_dt",
    y="incidents",
    color="country",
    title="Monthly Conflict Incidents: Sahel 2020–2025",
    labels={
        "year_month_dt": "Date",
        "incidents":     "Number of Incidents",
        "country":       "Country:",
    },
    color_discrete_map={
        "Burkina Faso": "#e74c3c",
        "Mali":         "#3498db",
        "Niger":        "#2ecc71",
    },
    template="plotly_dark",
)

fig.update_layout(
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=450,
)
fig.show()

## 2. Monthly Fatalities
---
Incidents and fatalities do not always follow the same pattern.  
A month with few incidents can have a very high death toll if those 
incidents involve massacres or large-scale battles.

This chart shows the **human cost** of conflict over time, per country.

In [7]:
fig2 = px.line(
    monthly,
    x="year_month_dt",
    y="fatalities",
    color="country",
    title="Monthly Fatalities: Sahel 2020–2025",
    labels={
        "year_month_dt": "Date",
        "fatalities":    "Fatalities",
        "country":       "Country",
    },
    color_discrete_map={
        "Burkina Faso": "#e74c3c",
        "Mali":         "#3498db",
        "Niger":        "#2ecc71",
    },
    template="plotly_dark",
)

# Fill under each line independently (not stacked)
fig2.update_traces(fill="tozeroy", opacity=0.4)

fig2.update_layout(
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=450,
)
fig2.show()

## 3. Breakdown by Event Type
---
Not all conflict events are equal. ACLED classifies events into 6 categories:

| Type | Description |
|---|---|
| **Battles** | Armed clashes between organized groups |
| **Violence against civilians** | Targeted attacks on non-combatants |
| **Explosions/Remote violence** | IEDs, airstrikes, artillery |
| **Protests** | Non-violent demonstrations |
| **Riots** | Violent demonstrations |
| **Strategic developments** | Agreements, arrests, non-violent changes |

This stacked bar chart shows **how the nature of conflict has evolved** over time.

In [9]:
# --- Monthly breakdown by event type (all countries combined)
monthly_type = (
    df.groupby(["year_month", "event_type"])
    .agg(incidents=("event_id_cnty", "count"))
    .reset_index()
)
monthly_type["year_month_dt"] = pd.to_datetime(monthly_type["year_month"])
monthly_type = monthly_type.sort_values("year_month_dt")

EVENT_COLORS = {
    "Battles":                       "#e74c3c",
    "Violence against civilians":    "#e67e22",
    "Explosions/Remote violence":    "#9b59b6",
    "Protests":                      "#3498db",
    "Riots":                         "#f1c40f",
    "Strategic developments":        "#2ecc71",
}

fig3 = px.bar(
    monthly_type,
    x="year_month_dt",
    y="incidents",
    color="event_type",
    title="Monthly Incidents by Event Type: Sahel 2020–2025",
    labels={
        "year_month_dt": "Date",
        "incidents":     "Number of Incidents",
        "event_type":    "Event Type :",
    },
    color_discrete_map=EVENT_COLORS,
    template="plotly_dark",
)

fig3.update_layout(
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=450,
    bargap=0.1,
)
fig3.show()

## 4. Anomaly Detection
---
Beyond visual inspection, we use a **statistical approach** to identify months 
with abnormally high conflict activity.

**Method: Z-score**  
The Z-score measures how many standard deviations a value is from the mean.  
A month is flagged as an anomaly if `|Z-score| > 1.5`.

$$Z = \frac{x - \mu}{\sigma}$$

These anomalous months often correspond to real events:
coups d'état, major military offensives, or political crises.

In [11]:
# --- Anomaly detection using Z-score on total monthly incidents
monthly_total = (
    df.groupby("year_month")
    .agg(incidents=("event_id_cnty", "count"))
    .reset_index()
)
monthly_total["year_month_dt"] = pd.to_datetime(monthly_total["year_month"])
monthly_total = monthly_total.sort_values("year_month_dt")

# Z-score: values beyond 1.5 std are flagged as anomalies
monthly_total["zscore"]    = stats.zscore(monthly_total["incidents"])
monthly_total["is_anomaly"] = monthly_total["zscore"].abs() > 1.5

normal   = monthly_total[~monthly_total["is_anomaly"]]
anomalies = monthly_total[monthly_total["is_anomaly"]]

print(f"Anomalous months detected: {len(anomalies)}")
print(anomalies[["year_month", "incidents", "zscore"]].to_string(index=False))

# --- Plot
fig4 = go.Figure()

# Normal months
fig4.add_trace(go.Scatter(
    x=normal["year_month_dt"],
    y=normal["incidents"],
    mode="lines+markers",
    name="Normal",
    line=dict(color="#3498db", width=2),
    marker=dict(size=5),
))

# Anomalous months
fig4.add_trace(go.Scatter(
    x=anomalies["year_month_dt"],
    y=anomalies["incidents"],
    mode="markers",
    name="Anomaly",
    marker=dict(color="#e74c3c", size=12, symbol="star"),
))

# Rolling average (3-month)
monthly_total["rolling_avg"] = monthly_total["incidents"].rolling(3, center=True).mean()
fig4.add_trace(go.Scatter(
    x=monthly_total["year_month_dt"],
    y=monthly_total["rolling_avg"],
    mode="lines",
    name="3-month Rolling Avg",
    line=dict(color="#2ecc71", width=2, dash="dash"),
))

fig4.update_layout(
    title="Anomaly Detection: Monthly Conflict Incidents",
    xaxis_title="Date",
    yaxis_title="Number of Incidents",
    template="plotly_dark",
    hovermode="x unified",
    height=450,
)
fig4.show()

Anomalous months detected: 6
year_month  incidents    zscore
   2020-02        195 -1.677551
   2020-12        182 -1.803935
   2021-02        172 -1.901153
   2021-03        199 -1.638664
   2022-10        585  2.113952
   2023-11        533  1.608419


## 5. Cross-Country Correlation
---
Do conflicts in one Sahel country influence conflict levels in neighboring countries?

We compute the **Pearson correlation** between monthly incident counts across 
the three countries. A high positive correlation suggests that conflict 
dynamics are regionally synchronized — likely driven by the same armed groups 
operating across borders (e.g., JNIM, ISWAP).

**Correlation scale:**
- `> 0.6` → Strong
- `0.3 – 0.6` → Moderate  
- `< 0.3` → Weak

In [12]:
# --- Pivot to wide format: one column per country
monthly_pivot = monthly.pivot(
    index="year_month_dt",
    columns="country",
    values="incidents"
).fillna(0)

# --- Correlation matrix
corr = monthly_pivot.corr()

print("Correlation matrix (monthly incidents):")
print(corr.round(3))

# --- Heatmap
fig5 = px.imshow(
    corr,
    title="Cross-Country Conflict Correlation",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    text_auto=".2f",
    template="plotly_dark",
)
fig5.update_layout(height=400)
fig5.show()

# --- Interpretation
print("\nInterpretation:")
for c1 in corr.columns:
    for c2 in corr.columns:
        if c1 < c2:
            r = corr.loc[c1, c2]
            strength = "strong" if abs(r) > 0.6 else "moderate" if abs(r) > 0.3 else "weak"
            direction = "positive" if r > 0 else "negative"
            print(f"  {c1} <-> {c2}: {r:.3f} ({strength} {direction} correlation)")

Correlation matrix (monthly incidents):
country       Burkina Faso   Mali  Niger
country                                 
Burkina Faso         1.000  0.486  0.309
Mali                 0.486  1.000  0.516
Niger                0.309  0.516  1.000



Interpretation:
  Burkina Faso <-> Mali: 0.486 (moderate positive correlation)
  Burkina Faso <-> Niger: 0.309 (moderate positive correlation)
  Mali <-> Niger: 0.516 (moderate positive correlation)


## 6. Trend Model & 6-Month Forecast
---
We fit a **linear regression** on the monthly incident time series to capture 
the overall conflict trajectory across the Sahel.

**Model:** $y_t = \alpha + \beta t + \epsilon$

Where:
- $y_t$ = number of incidents in month $t$
- $\beta$ = trend slope (positive = escalation, negative = de-escalation)
- $R^2$ = goodness of fit

We also overlay **rolling averages** (3, 6, 12 months) to smooth short-term 
volatility and reveal the underlying trend.

> Note: This is a simple linear trend model. In Phase 3 of this project,  
> we will replace this with a more robust Bayesian forecasting model that  
> accounts for uncertainty and structural breaks in the data.

In [13]:
# --- Trend model on total Sahel incidents
monthly_total = monthly_total.sort_values("year_month_dt").reset_index(drop=True)
monthly_total["t"] = np.arange(len(monthly_total))  # numeric time index

# Linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(
    monthly_total["t"],
    monthly_total["incidents"]
)

monthly_total["linear_trend"] = intercept + slope * monthly_total["t"]

print(f"Linear trend:")
print(f"  Slope     : {slope:+.2f} incidents/month")
print(f"  R²        : {r_value**2:.3f}")
print(f"  p-value   : {p_value:.4f}")
print(f"  Significant: {'Yes' if p_value < 0.05 else 'No'}")

# --- 6-month forecast
last_t    = monthly_total["t"].max()
last_date = monthly_total["year_month_dt"].max()
future_t  = np.arange(last_t + 1, last_t + 7)
future_dates = pd.date_range(
    start=last_date + pd.DateOffset(months=1),
    periods=6,
    freq="MS"
)
future_trend = intercept + slope * future_t

# --- Rolling averages
monthly_total["roll_3"]  = monthly_total["incidents"].rolling(3,  center=False).mean()
monthly_total["roll_6"]  = monthly_total["incidents"].rolling(6,  center=False).mean()
monthly_total["roll_12"] = monthly_total["incidents"].rolling(12, center=False).mean()

# --- Plot
fig6 = go.Figure()

# Raw data
fig6.add_trace(go.Bar(
    x=monthly_total["year_month_dt"],
    y=monthly_total["incidents"],
    name="Monthly incidents",
    marker_color="rgba(52, 152, 219, 0.4)",
))

# Rolling averages
for roll, color, name in [
    ("roll_3",  "#f1c40f", "3-month MA"),
    ("roll_6",  "#e67e22", "6-month MA"),
    ("roll_12", "#e74c3c", "12-month MA"),
]:
    fig6.add_trace(go.Scatter(
        x=monthly_total["year_month_dt"],
        y=monthly_total[roll],
        mode="lines",
        name=name,
        line=dict(color=color, width=2),
    ))

# Linear trend (historical)
fig6.add_trace(go.Scatter(
    x=monthly_total["year_month_dt"],
    y=monthly_total["linear_trend"],
    mode="lines",
    name="Linear trend",
    line=dict(color="#2ecc71", width=2, dash="dot"),
))

# Forecast
fig6.add_trace(go.Scatter(
    x=future_dates,
    y=future_trend,
    mode="lines+markers",
    name="6-month forecast",
    line=dict(color="#2ecc71", width=2, dash="dash"),
    marker=dict(size=8, symbol="diamond"),
))

# Forecast zone
fig6.add_vrect(
    x0=future_dates[0], x1=future_dates[-1],
    fillcolor="rgba(46, 204, 113, 0.05)",
    line_width=0,
    annotation_text="Forecast",
    annotation_position="top left",
)

fig6.update_layout(
    title="Conflict Trend Analysis & 6-Month Forecast — Sahel",
    xaxis_title="Date",
    yaxis_title="Number of Incidents",
    template="plotly_dark",
    hovermode="x unified",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig6.show()

Linear trend:
  Slope     : +3.36 incidents/month
  R²        : 0.354
  p-value   : 0.0000
  Significant: Yes


## 7. Saving Temporal Data
---
We save the monthly aggregations to CSV files for reuse in the  
**Streamlit dashboard** (Module 4) without recomputing everything from scratch.

In [15]:
 # Save monthly aggregations for use in the Streamlit dashboard
monthly.to_csv("../data/processed/monthly_by_country.csv",    index=False)
monthly_type.to_csv("../data/processed/monthly_by_type.csv",  index=False)
monthly_total.to_csv("../data/processed/monthly_total.csv",   index=False)

print("Temporal data saved:")
print("  -> data/processed/monthly_by_country.csv")
print("  -> data/processed/monthly_by_type.csv")
print("  -> data/processed/monthly_total.csv")

Temporal data saved:
  -> data/processed/monthly_by_country.csv
  -> data/processed/monthly_by_type.csv
  -> data/processed/monthly_total.csv


## verifying why my data range stop on march 2025

In [17]:
import os
import requests

# Credentials are loaded from environment variables / .env
USERNAME = os.getenv("ACLED_USERNAME")
PASSWORD = os.getenv("ACLED_PASSWORD")

response = requests.post(
    "https://acleddata.com/oauth/token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={"username": USERNAME, "password": PASSWORD,
          "grant_type": "password", "client_id": "acled"},
)
token = response.json()["access_token"]
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Check most recent available date
url = "https://acleddata.com/api/acled/read?_format=json&country=Burkina Faso&limit=1&fields=event_id_cnty|event_date|country"
r = requests.get(url, headers=headers)
data = r.json().get("data", [])
print(f"Most recent event available: {data[0]['event_date'] if data else 'No data'}")

Most recent event available: 2019-12-27


In [18]:
import os
import requests

# Credentials are loaded from environment variables / .env
USERNAME = os.getenv("ACLED_USERNAME")
PASSWORD = os.getenv("ACLED_PASSWORD")

response = requests.post(
    "https://acleddata.com/oauth/token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={"username": USERNAME, "password": PASSWORD,
          "grant_type": "password", "client_id": "acled"},
)
token = response.json()["access_token"]
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Sort by event_date descending to get the most recent event
url = (
    "https://acleddata.com/api/acled/read?_format=json"
    "&country=Burkina Faso"
    "&limit=1"
    "&fields=event_id_cnty|event_date|country"
    "&order_by=event_date"
    "&order=desc"
)
r = requests.get(url, headers=headers)
data = r.json().get("data", [])
print(f"Most recent event available : {data[0]['event_date'] if data else 'No data'}")
print(f"Full response               : {r.json()}")

Most recent event available : 2019-12-27
Full response               : {'status': 200, 'success': True, 'count': 1, 'total_count': 12292, 'messages': [], 'data': [{'event_id_cnty': 'BFO3375', 'event_date': '2019-12-27', 'country': 'Burkina Faso'}], 'filename': 'results.json', 'data_query_restrictions': {'countries': [], 'event_types': [], 'regions': [], 'history': [], 'recency': [], 'date_recency': {'quantity': 12, 'unit': 'Months', 'description': '12 Months old', 'timestamp': 1743234884, 'date': '2025-03-29'}}}
